# 🚀 crdt-merge v0.6.0 — A100 GPU Stress Test Notebook

**Purpose:** Comprehensive stress test of every crdt-merge 0.6.0 subsystem, optimized for NVIDIA A100 GPU on Google Colab.

**Sections:** 21 test sections covering Core CRDTs, Merge Strategies, DataFrame merge/diff, Streaming, Provenance, JSON merge, Dedup, Wire Protocol, Probabilistic, CRDT Verification, Clocks, Schema Evolution, Merkle Trees, Gossip, Arrow, Async, Parallel, Integration Pipelines, Results Export, and Visualization.

**Adaptive scaling:** Automatically detects hardware tier (A100 / HIGH / STANDARD) and scales workloads accordingly.

In [ ]:
# ── Section 1: Setup ──────────────────────────────────────────────────────────
import subprocess, sys, os, time, json, datetime, math, random, hashlib, string
from collections import defaultdict

# Install dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "crdt-merge==0.6.0", "pandas", "pyarrow", "matplotlib", "nest_asyncio"])

import pandas as pd
import numpy as np

# ── Hardware detection ────────────────────────────────────────────────────────
GPU_NAME = "NONE"
GPU_MEM_GB = 0
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        text=True).strip()
    parts = out.split(",")
    GPU_NAME = parts[0].strip()
    GPU_MEM_GB = int(parts[1].strip()) / 1024
except Exception:
    pass

import psutil
RAM_GB = psutil.virtual_memory().total / (1024**3)
CPU_COUNT = os.cpu_count() or 1

# ── Scale multiplier ─────────────────────────────────────────────────────────
if "A100" in GPU_NAME:
    SM = 10
    TIER = "A100"
elif GPU_MEM_GB >= 10 or RAM_GB >= 24:
    SM = 3
    TIER = "HIGH"
else:
    SM = 1
    TIER = "STANDARD"

print(f"🖥️  GPU: {GPU_NAME} ({GPU_MEM_GB:.1f} GB)")
print(f"💾  RAM: {RAM_GB:.1f} GB  |  CPUs: {CPU_COUNT}")
print(f"📊  Tier: {TIER}  |  Scale Multiplier (SM): {SM}×")

# ── Results accumulator ──────────────────────────────────────────────────────
ALL_RESULTS = {}

def bench(label, fn, *args, **kwargs):
    """Benchmark a callable, store result in ALL_RESULTS."""
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = time.perf_counter() - t0
    ALL_RESULTS[label] = {"elapsed_s": round(elapsed, 6), "status": "OK"}
    print(f"  ✅ {label}: {elapsed:.4f}s")
    return result

def scale_bench(label, fn, sizes):
    """Run fn(n) for each n in sizes, record throughput."""
    records = []
    for n in sizes:
        t0 = time.perf_counter()
        fn(n)
        elapsed = time.perf_counter() - t0
        throughput = n / elapsed if elapsed > 0 else float('inf')
        records.append({"n": n, "elapsed_s": round(elapsed, 6),
                        "throughput": round(throughput, 2)})
        print(f"  📈 {label} n={n:,}: {elapsed:.4f}s ({throughput:,.0f} ops/s)")
    ALL_RESULTS[label + "_scale"] = records
    return records

print("\n✅ Setup complete.")

## 2. Core CRDT Types

Test GCounter, PNCounter, LWWRegister, LWWMap, ORSet — verifying merge commutativity and idempotency.

In [ ]:
# ── Section 2: Core CRDT Types ────────────────────────────────────────────────
import datetime as dt
from crdt_merge import GCounter, PNCounter, LWWRegister, LWWMap, ORSet

print("── GCounter ──")
gc1 = GCounter("A")
gc1.increment("A", amount=5)
gc2 = GCounter("B")
gc2.increment("B", amount=3)
merged_gc = gc1.merge(gc2)
assert merged_gc.value == 8, f"Expected 8 got {merged_gc.value}"
# Commutativity
merged_gc2 = gc2.merge(gc1)
assert merged_gc.value == merged_gc2.value, "GCounter merge not commutative!"
# Idempotency
merged_gc3 = merged_gc.merge(gc1)
assert merged_gc3.value == merged_gc.value, "GCounter merge not idempotent!"
bench("gcounter_merge", gc1.merge, gc2)
print(f"  GCounter merged value: {merged_gc.value}")

print("\n── PNCounter ──")
pc1 = PNCounter()
pc1.increment("X", amount=10)
pc1.decrement("X", amount=3)
pc2 = PNCounter()
pc2.increment("X", amount=5)
merged_pc = pc1.merge(pc2)
print(f"  PNCounter merged value: {merged_pc.value}")
bench("pncounter_merge", pc1.merge, pc2)

print("\n── LWWRegister ──")
lr1 = LWWRegister("old_value", dt.datetime(2025, 1, 1))
lr2 = LWWRegister("new_value", dt.datetime(2026, 1, 1))
merged_lr = lr1.merge(lr2)
assert merged_lr.value == "new_value"
# Commutativity
merged_lr2 = lr2.merge(lr1)
assert merged_lr2.value == merged_lr.value, "LWWRegister merge not commutative!"
bench("lwwregister_merge", lr1.merge, lr2)
print(f"  LWWRegister merged value: {merged_lr.value}")

print("\n── LWWMap ──")
lm1 = LWWMap()
lm1.set("color", "red", timestamp=1000000.0, node_id="n1")
lm2 = LWWMap()
lm2.set("color", "blue", timestamp=2000000.0, node_id="n2")
merged_lm = lm1.merge(lm2)
result_dict = merged_lm.to_dict()
print(f"  LWWMap merged: {result_dict}")
bench("lwwmap_merge", lm1.merge, lm2)

print("\n── ORSet ──")
os1 = ORSet()
os1.add("apple")
os1.add("banana")
os2 = ORSet()
os2.add("banana")
os2.add("cherry")
merged_os = os1.merge(os2)
print(f"  ORSet merged value: {merged_os.value}")
# Commutativity
merged_os2 = os2.merge(os1)
assert merged_os.value == merged_os2.value, "ORSet merge not commutative!"
bench("orset_merge", os1.merge, os2)

print("\n── Scale: GCounter chains ──")
def gc_chain(n):
    gc = GCounter("stress")
    for i in range(n):
        gc.increment("stress", amount=1)
    return gc

sizes = [1000 * SM, 10000 * SM, 50000 * SM]
scale_bench("gcounter_increment", gc_chain, sizes)

print("\n✅ Section 2 complete.")

## 3. Merge Strategies

Test all 8 strategies: LWW, MaxWins, MinWins, LongestWins, Priority, Concat, UnionSet, Custom.
Use MergeSchema with `merge()` and numeric epoch timestamps.

In [ ]:
# ── Section 3: Merge Strategies ───────────────────────────────────────────────
from crdt_merge import (MergeSchema, LWW, MaxWins, MinWins, LongestWins,
                        Priority, Concat, UnionSet, Custom, merge)

now = time.time()
ts1 = now - 100.0
ts2 = now

df_a = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "score": [80, 90, 70],
    "tag": ["a,b", "c", "d,e"],
    "level": ["low", "medium", "high"],
    "ts": [ts1, ts1, ts1]
})
df_b = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alicia", "Bobby McBobface", "Charles"],
    "score": [95, 85, 75],
    "tag": ["b,c", "c,d", "e,f"],
    "level": ["high", "low", "medium"],
    "ts": [ts2, ts2, ts2]
})

# Test each strategy individually
print("── Individual Strategies ──")
strategies = {
    "LWW": LWW(),
    "MaxWins": MaxWins(),
    "MinWins": MinWins(),
    "LongestWins": LongestWins(),
    "Priority": Priority(levels=["low", "medium", "high"]),
    "Concat": Concat(separator=" | ", dedup=True),
    "UnionSet": UnionSet(separator=","),
    "Custom": Custom(fn=lambda a, b: max(a, b) if isinstance(a, (int, float)) else a),
}
for name, strat in strategies.items():
    print(f"  Strategy: {name} -> {strat}")

# MergeSchema with multiple strategies
schema = MergeSchema(default=LWW(), score=MaxWins(), name=LongestWins(),
                     level=Priority(levels=["low", "medium", "high"]))

result = bench("merge_with_schema", merge, df_a, df_b, key="id",
               timestamp_col="ts", schema=schema)
print(f"  Schema merge result shape: {result.shape}")
print(result.to_string())

# Scale merge + schema
print("\n── Scale: merge with schema ──")
def scaled_merge(n):
    a = pd.DataFrame({
        "id": range(n),
        "val": np.random.randint(0, 100, n),
        "ts": [ts1] * n
    })
    b = pd.DataFrame({
        "id": range(n),
        "val": np.random.randint(0, 100, n),
        "ts": [ts2] * n
    })
    s = MergeSchema(default=LWW(), val=MaxWins())
    return merge(a, b, key="id", timestamp_col="ts", schema=s)

sizes = [1000 * SM, 5000 * SM, 20000 * SM]
scale_bench("merge_schema", scaled_merge, sizes)

print("\n✅ Section 3 complete.")

## 4. DataFrame Merge & Diff

Test `merge()` and `diff()` with key-based joining, prefer modes. Scale to 1M×SM rows.

In [ ]:
# ── Section 4: DataFrame merge + diff ─────────────────────────────────────────
from crdt_merge import merge, diff

now = time.time()
ts_old = now - 500.0
ts_new = now

# Basic merge
df_a = pd.DataFrame({"id": [1, 2, 3], "val": ["a", "b", "c"], "ts": [ts_old]*3})
df_b = pd.DataFrame({"id": [2, 3, 4], "val": ["B", "C", "D"], "ts": [ts_new]*3})

merged = bench("df_merge_basic", merge, df_a, df_b, key="id", timestamp_col="ts")
print(f"  Merged shape: {merged.shape}")
print(merged)

# Diff
d = bench("df_diff_basic", diff, df_a, df_b, key="id")
print(f"\n  Diff result: {d}")

# Prefer modes
for prefer in ['latest', 'left', 'right']:
    r = merge(df_a, df_b, key="id", timestamp_col="ts", prefer=prefer)
    print(f"  prefer='{prefer}' -> {len(r)} rows")

# Scale to 1M * SM rows
print("\n── Scale: merge to large rows ──")
def large_merge(n):
    a = pd.DataFrame({
        "id": range(n),
        "value": np.random.randn(n),
        "category": np.random.choice(["X", "Y", "Z"], n),
        "ts": [ts_old] * n
    })
    b = pd.DataFrame({
        "id": range(n // 2, n + n // 2),
        "value": np.random.randn(n),
        "category": np.random.choice(["X", "Y", "Z"], n),
        "ts": [ts_new] * n
    })
    return merge(a, b, key="id", timestamp_col="ts")

sizes = [10000 * SM, 100000 * SM, 1000000 * SM]
scale_bench("df_merge_large", large_merge, sizes)

print("\n✅ Section 4 complete.")

## 5. Streaming Merge

Test `merge_stream()` with two iterables, consuming via `list()`. Scale with dict lists.

In [ ]:
# ── Section 5: Streaming ──────────────────────────────────────────────────────
from crdt_merge import merge_stream

src_a = [{"id": 1, "val": "a"}, {"id": 2, "val": "b"}, {"id": 3, "val": "c"}]
src_b = [{"id": 2, "val": "B"}, {"id": 3, "val": "C"}, {"id": 4, "val": "D"}]

gen = merge_stream(iter(src_a), iter(src_b), key="id")
result = bench("stream_merge_basic", list, gen)
print(f"  Stream merge result: {len(result)} batches")
for batch in result:
    if isinstance(batch, list):
        print(f"    Batch size: {len(batch)}")
    else:
        print(f"    Item: {batch}")

# Scale streaming
print("\n── Scale: streaming merge ──")
def stream_scale(n):
    a = [{"id": i, "val": f"a_{i}"} for i in range(n)]
    b = [{"id": i + n // 2, "val": f"b_{i}"} for i in range(n)]
    gen = merge_stream(iter(a), iter(b), key="id")
    return list(gen)

sizes = [5000 * SM, 20000 * SM, 100000 * SM]
scale_bench("stream_merge", stream_scale, sizes)

print("\n✅ Section 5 complete.")

## 6. Provenance Tracking

Test `merge_with_provenance()` and `export_provenance()`. Timestamps MUST be numeric epoch floats.

In [ ]:
# ── Section 6: Provenance ─────────────────────────────────────────────────────
from crdt_merge import merge_with_provenance, export_provenance

now = time.time()
ts_a = now - 300.0
ts_b = now

df_a = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "score": [80, 90, 70],
    "ts": [ts_a, ts_a, ts_a]
})
df_b = pd.DataFrame({
    "id": [2, 3, 4],
    "name": ["Bobby", "Charles", "Diana"],
    "score": [95, 65, 88],
    "ts": [ts_b, ts_b, ts_b]
})

result_df, prov_log = bench("provenance_merge", merge_with_provenance,
                            df_a, df_b, key="id", timestamp_col="ts")
print(f"  Result rows: {len(result_df)}")
print(f"  Total conflicts: {prov_log.total_conflicts}")

# Export provenance
json_str = bench("provenance_export_json", export_provenance, prov_log, format="json")
csv_str = bench("provenance_export_csv", export_provenance, prov_log, format="csv")
print(f"  JSON export length: {len(json_str)} chars")
print(f"  CSV export length: {len(csv_str)} chars")

# Scale
print("\n── Scale: provenance merge ──")
def prov_scale(n):
    a = pd.DataFrame({
        "id": range(n), "value": np.random.randn(n),
        "ts": [ts_a] * n
    })
    b = pd.DataFrame({
        "id": range(n // 2, n + n // 2), "value": np.random.randn(n),
        "ts": [ts_b] * n
    })
    r, p = merge_with_provenance(a, b, key="id", timestamp_col="ts")
    return p

sizes = [5000 * SM, 20000 * SM, 50000 * SM]
scale_bench("provenance", prov_scale, sizes)

print("\n✅ Section 6 complete.")

## 7. JSON Merge

Test `merge_dicts()` and `merge_json_lines()`. Scale both.

In [ ]:
# ── Section 7: JSON Merge ─────────────────────────────────────────────────────
from crdt_merge import merge_dicts, merge_json_lines

# merge_dicts
dict_a = {"name": "Alice", "score": 80, "tags": ["python"]}
dict_b = {"name": "Alicia", "score": 95, "tags": ["rust"]}
result = bench("merge_dicts_basic", merge_dicts, dict_a, dict_b)
print(f"  merge_dicts result: {result}")

# merge_json_lines
lines_a = [
    {"id": 1, "name": "Alice"},
    {"id": 2, "name": "Bob"}
]
lines_b = [
    {"id": 2, "name": "Bobby"},
    {"id": 3, "name": "Charlie"}
]
result_jl = bench("merge_json_lines_basic", merge_json_lines, lines_a, lines_b, key="id")
print(f"  merge_json_lines result: {result_jl}")

# Scale merge_dicts
print("\n── Scale: merge_dicts ──")
def scale_merge_dicts(n):
    a = {f"key_{i}": f"val_a_{i}" for i in range(n)}
    b = {f"key_{i}": f"val_b_{i}" for i in range(n)}
    return merge_dicts(a, b)

sizes = [1000 * SM, 10000 * SM, 50000 * SM]
scale_bench("merge_dicts", scale_merge_dicts, sizes)

# Scale merge_json_lines
print("\n── Scale: merge_json_lines ──")
def scale_json_lines(n):
    la = [{"id": i, "val": f"a{i}"} for i in range(n)]
    lb = [{"id": i + n // 2, "val": f"b{i}"} for i in range(n)]
    return merge_json_lines(la, lb, key="id")

sizes = [1000 * SM, 5000 * SM, 20000 * SM]
scale_bench("merge_json_lines", scale_json_lines, sizes)

print("\n✅ Section 7 complete.")

## 8. Deduplication

Test `dedup()` (List[str]), `dedup_records()`, `DedupIndex`, `MinHashDedup`. Scale dedup on string lists.

In [ ]:
# ── Section 8: Deduplication ───────────────────────────────────────────────────
from crdt_merge import dedup, dedup_records, DedupIndex, MinHashDedup

# dedup on list of strings
items = ["hello", "world", "hello", "foo", "world", "bar"]
unique_list, removed_indices = bench("dedup_strings", dedup, items)
print(f"  dedup: {unique_list}, removed indices: {removed_indices}")

# dedup_records
records = [{"id": 1, "name": "A"}, {"id": 1, "name": "A"}, {"id": 2, "name": "B"}]
unique_recs, removed_count = bench("dedup_records", dedup_records, records)
print(f"  dedup_records: {len(unique_recs)} unique, {removed_count} removed")

# DedupIndex
di = DedupIndex("node1")
r1 = di.add_exact("hello world")
r2 = di.add_exact("hello world")  # should be False (duplicate)
r3 = di.add_exact("something new")
print(f"  DedupIndex: add1={r1}, add2(dup)={r2}, add3={r3}, size={di.size}")
bench("dedup_index_ops", lambda: di.add_exact("another text"))

# MinHashDedup
mh = MinHashDedup(num_hashes=128, threshold=0.5)
is_new1 = mh.add("k1", "The quick brown fox jumps over the lazy dog")
is_new2 = mh.add("k2", "The quick brown fox leaps over the lazy dog")
is_new3 = mh.add("k3", "Completely different text about machine learning")
print(f"  MinHashDedup: new1={is_new1}, new2(similar)={is_new2}, new3={is_new3}")

# MinHashDedup.dedup
items_for_dedup = [
    {"key": "a", "text": "hello world foo bar"},
    {"key": "b", "text": "hello world foo baz"},
    {"key": "c", "text": "totally different content"},
]
deduped = mh.dedup(items_for_dedup, text_fn=lambda x: x["text"])
print(f"  MinHashDedup.dedup: {len(items_for_dedup)} -> {len(deduped)}")
bench("minhash_dedup", mh.dedup, items_for_dedup, lambda x: x["text"])

# Scale dedup on string lists
print("\n── Scale: dedup strings ──")
def scale_dedup(n):
    items = [f"item_{i % (n // 3 + 1)}" for i in range(n)]
    return dedup(items)

sizes = [10000 * SM, 50000 * SM, 200000 * SM]
scale_bench("dedup_strings", scale_dedup, sizes)

print("\n✅ Section 8 complete.")

## 9. Wire Protocol

Test `serialize` / `deserialize` for all 14 CRDT types. Scale `serialize_batch`.

In [ ]:
# ── Section 9: Wire Protocol ──────────────────────────────────────────────────
from crdt_merge import (
    GCounter, PNCounter, LWWRegister, LWWMap, ORSet,
    MergeableBloom, MergeableCMS, MergeableHLL,
    VectorClock, DottedVersionVector, MerkleTree, GossipState,
    serialize, deserialize, serialize_batch, deserialize_batch, peek_type, wire_size
)
from crdt_merge.gossip import GossipEntry
from crdt_merge import SchemaChange
import datetime as dt

# Create instances of all 14 types
gc = GCounter("n1"); gc.increment("n1", amount=5)
pc = PNCounter(); pc.increment("n1", amount=3); pc.decrement("n1", amount=1)
lr = LWWRegister("test_value", dt.datetime(2026, 1, 1))
lm = LWWMap(); lm.set("k", "v", timestamp=1000000.0, node_id="n1")
oset = ORSet(); oset.add("x"); oset.add("y")
bl = MergeableBloom(capacity=100, fp_rate=0.01); bl.add("item1")
cms = MergeableCMS(width=100, depth=5); cms.add("item1")
hll = MergeableHLL(precision=12); hll.add("item1")
vc = VectorClock(); vc.increment("A"); vc.increment("B")
dvv = DottedVersionVector(base=VectorClock(), dot=("n1", 1))
ge = GossipEntry(key="k", value="v", clock=VectorClock(), tombstone=False)
sc = SchemaChange(column="col1", change_type="added", old_type=None,
                  new_type="string", resolved_type="string", default_value=None)
mt = MerkleTree.from_records([{"id": 1, "v": "a"}, {"id": 2, "v": "b"}], key="id")
gs = GossipState("n1"); gs.update("key1", "val1")

all_types = {
    "GCounter": gc, "PNCounter": pc, "LWWRegister": lr, "LWWMap": lm,
    "ORSet": oset, "MergeableBloom": bl, "MergeableCMS": cms,
    "MergeableHLL": hll, "VectorClock": vc, "DottedVersionVector": dvv,
    "GossipEntry": ge, "SchemaChange": sc, "MerkleTree": mt, "GossipState": gs
}

print("── Serialize / Deserialize each type ──")
for name, obj in all_types.items():
    data = serialize(obj)
    restored = deserialize(data)
    sz = wire_size(data)
    tp = peek_type(data)
    print(f"  {name}: wire_size={sz} bytes, peek_type={tp}")
    ALL_RESULTS[f"wire_{name}"] = {"wire_size": sz, "peek_type": str(tp), "status": "OK"}

# Batch serialize
print("\n── serialize_batch ──")
batch_items = list(all_types.values())
batch_data = bench("serialize_batch", serialize_batch, batch_items)
restored_batch = bench("deserialize_batch", deserialize_batch, batch_data)
print(f"  Batch: {len(batch_items)} items -> {len(batch_data)} bytes -> {len(restored_batch)} restored")

# Scale serialize_batch
print("\n── Scale: serialize_batch ──")
def scale_wire(n):
    items = []
    for i in range(n):
        g = GCounter(f"n{i}")
        g.increment(f"n{i}", amount=1)
        items.append(g)
    return serialize_batch(items)

sizes = [100 * SM, 1000 * SM, 5000 * SM]
scale_bench("serialize_batch", scale_wire, sizes)

print("\n✅ Section 9 complete.")

## 10. Probabilistic Data Structures

Test MergeableBloom, MergeableCMS, MergeableHLL — add, query, merge. Scale bloom and HLL adds.

In [ ]:
# ── Section 10: Probabilistic ─────────────────────────────────────────────────
from crdt_merge import MergeableBloom, MergeableCMS, MergeableHLL

# Bloom filter
print("── MergeableBloom ──")
bl1 = MergeableBloom(capacity=10000, fp_rate=0.01)
bl2 = MergeableBloom(capacity=10000, fp_rate=0.01)
for i in range(100):
    bl1.add(f"item_{i}")
for i in range(50, 150):
    bl2.add(f"item_{i}")
assert bl1.contains("item_0")
assert bl1.contains("item_99")
merged_bl = bench("bloom_merge", bl1.merge, bl2)
print(f"  Bloom: 'item_0' in merged = {merged_bl.contains('item_0')}")
print(f"  Bloom: 'item_149' in merged = {merged_bl.contains('item_149')}")

# CMS
print("\n── MergeableCMS ──")
cms1 = MergeableCMS(width=200, depth=7)
cms2 = MergeableCMS(width=200, depth=7)
for i in range(50):
    cms1.add(f"word_{i % 10}")
for i in range(50):
    cms2.add(f"word_{i % 10}")
merged_cms = bench("cms_merge", cms1.merge, cms2)
est = merged_cms.estimate("word_0")
print(f"  CMS estimate('word_0') after merge: {est}")

# HLL
print("\n── MergeableHLL ──")
hll1 = MergeableHLL(precision=14)
hll2 = MergeableHLL(precision=14)
for i in range(5000):
    hll1.add(f"user_{i}")
for i in range(3000, 8000):
    hll2.add(f"user_{i}")
merged_hll = bench("hll_merge", hll1.merge, hll2)
count = merged_hll.cardinality()
print(f"  HLL merged count: {count} (expected ~8000)")

# Scale bloom adds
print("\n── Scale: Bloom adds ──")
def bloom_add_scale(n):
    b = MergeableBloom(capacity=n * 2, fp_rate=0.01)
    for i in range(n):
        b.add(f"item_{i}")
    return b

sizes = [10000 * SM, 50000 * SM, 200000 * SM]
scale_bench("bloom_add", bloom_add_scale, sizes)

# Scale HLL adds
print("\n── Scale: HLL adds ──")
def hll_add_scale(n):
    h = MergeableHLL(precision=14)
    for i in range(n):
        h.add(f"user_{i}")
    return h

scale_bench("hll_add", hll_add_scale, sizes)

print("\n✅ Section 10 complete.")

## 11. CRDT Verification

Test `verify_crdt()` from submodule on GCounter, PNCounter, LWWRegister, ORSet, VectorClock. Also `@verified_merge`.

In [ ]:
# ── Section 11: CRDT Verification ─────────────────────────────────────────────
from crdt_merge.verify import verify_crdt
from crdt_merge import (GCounter, PNCounter, LWWRegister, ORSet, VectorClock,
                        verified_merge)
import datetime as dt, random

# GCounter verification
def gc_gen():
    gc = GCounter("test")
    gc.increment("test", amount=random.randint(1, 100))
    return gc

def gc_merge(a, b):
    return a.merge(b)

result_gc = bench("verify_gcounter", verify_crdt, gc_merge, gc_gen, trials=10)
print(f"  GCounter: comm={result_gc.commutativity.passed}, "
      f"assoc={result_gc.associativity.passed}, idem={result_gc.idempotency.passed}")

# PNCounter
def pnc_gen():
    pc = PNCounter()
    pc.increment("test", amount=random.randint(1, 50))
    pc.decrement("test", amount=random.randint(1, 20))
    return pc

result_pnc = bench("verify_pncounter", verify_crdt, lambda a, b: a.merge(b), pnc_gen, trials=10)
print(f"  PNCounter: comm={result_pnc.commutativity.passed}, "
      f"assoc={result_pnc.associativity.passed}, idem={result_pnc.idempotency.passed}")

# LWWRegister
def lr_gen():
    return LWWRegister(f"val_{random.randint(0,100)}",
                       dt.datetime(2026, 1, random.randint(1,28)))

result_lr = bench("verify_lwwregister", verify_crdt, lambda a, b: a.merge(b), lr_gen, trials=10)
print(f"  LWWRegister: comm={result_lr.commutativity.passed}, "
      f"assoc={result_lr.associativity.passed}, idem={result_lr.idempotency.passed}")

# ORSet
def orset_gen():
    o = ORSet()
    for _ in range(random.randint(1, 5)):
        o.add(f"elem_{random.randint(0, 10)}")
    return o

result_os = bench("verify_orset", verify_crdt, lambda a, b: a.merge(b), orset_gen, trials=10)
print(f"  ORSet: comm={result_os.commutativity.passed}, "
      f"assoc={result_os.associativity.passed}, idem={result_os.idempotency.passed}")

# VectorClock
def vc_gen():
    v = VectorClock()
    for _ in range(random.randint(1, 5)):
        v.increment(f"node_{random.randint(0, 3)}")
    return v

result_vc = bench("verify_vectorclock", verify_crdt, lambda a, b: a.merge(b), vc_gen, trials=10)
print(f"  VectorClock: comm={result_vc.commutativity.passed}, "
      f"assoc={result_vc.associativity.passed}, idem={result_vc.idempotency.passed}")

# @verified_merge decorator
@verified_merge(gen_fn=gc_gen, trials=10)
def my_merge(a, b):
    return a.merge(b)

test_res = my_merge(gc_gen(), gc_gen())
print(f"\n  @verified_merge test: {test_res}")

print("\n✅ Section 11 complete.")

## 12. Clocks — VectorClock & DottedVersionVector

Test increment, compare (→ Ordering enum), merge, to_dict. Scale vc ops.

In [ ]:
# ── Section 12: Clocks ────────────────────────────────────────────────────────
from crdt_merge import VectorClock, DottedVersionVector, Ordering

print("── VectorClock ──")
vc1 = VectorClock()
vc1.increment("A")
vc1.increment("A")
vc1.increment("B")
vc2 = VectorClock()
vc2.increment("A")
vc2.increment("C")

ordering = vc1.compare(vc2)
print(f"  vc1 vs vc2: {ordering}")
print(f"  vc1 dict: {vc1.to_dict()}")
print(f"  vc2 dict: {vc2.to_dict()}")

merged_vc = bench("vc_merge", vc1.merge, vc2)
print(f"  Merged VC: {merged_vc.to_dict()}")

# Test Ordering values
vc_equal = VectorClock()
vc_equal.increment("A")
vc_copy = VectorClock()
vc_copy.increment("A")
assert vc_equal.compare(vc_copy) == Ordering.EQUAL
print(f"  Equal clocks: {vc_equal.compare(vc_copy)}")

print("\n── DottedVersionVector ──")
dvv1 = DottedVersionVector(base=VectorClock(), dot=("node_A", 1))
dvv2 = DottedVersionVector(base=VectorClock(), dot=("node_B", 1))
merged_dvv = bench("dvv_merge", dvv1.merge, dvv2)
print(f"  DVV merged: {merged_dvv}")

# Scale VC ops
print("\n── Scale: VectorClock ops ──")
def vc_scale(n):
    vc = VectorClock()
    for i in range(n):
        vc.increment(f"node_{i % 10}")
    return vc

sizes = [10000 * SM, 50000 * SM, 200000 * SM]
scale_bench("vectorclock_ops", vc_scale, sizes)

print("\n✅ Section 12 complete.")

## 13. Schema Evolution

Test `evolve_schema()` and `check_compatibility()` for detecting and resolving schema changes.

In [ ]:
# ── Section 13: Schema Evolution ──────────────────────────────────────────────
from crdt_merge import evolve_schema, check_compatibility, SchemaChange

old = {"id": "int", "name": "string", "score": "float"}
new = {"id": "int", "name": "string", "score": "double", "email": "string"}

result = bench("evolve_schema", evolve_schema, old, new)
print(f"  Resolved schema: {result.resolved_schema}")
print(f"  Changes: {result.changes}")
print(f"  Is compatible: {result.is_compatible}")

# check_compatibility
compat = bench("check_compatibility", check_compatibility, old, new)
print(f"  Compatibility result: {compat}")

# Scale
print("\n── Scale: schema evolution ──")
def schema_scale(n):
    old_s = {f"col_{i}": "string" for i in range(n)}
    new_s = {f"col_{i}": ("int" if i % 3 == 0 else "string") for i in range(n)}
    new_s[f"col_new_{n}"] = "float"
    return evolve_schema(old_s, new_s)

sizes = [100 * SM, 500 * SM, 2000 * SM]
scale_bench("schema_evolution", schema_scale, sizes)

print("\n✅ Section 13 complete.")

## 14. Merkle Trees

Test `MerkleTree.from_records()`, `.root_hash` (property), `.size`, `merkle_diff()`. Scale.

In [ ]:
# ── Section 14: Merkle Trees ──────────────────────────────────────────────────
from crdt_merge import MerkleTree, merkle_diff

recs_a = [{"id": i, "val": f"a_{i}"} for i in range(100)]
recs_b = [{"id": i, "val": f"a_{i}" if i < 90 else f"b_{i}"} for i in range(100)]

mt_a = bench("merkle_build_a", MerkleTree.from_records, recs_a, key="id")
mt_b = bench("merkle_build_b", MerkleTree.from_records, recs_b, key="id")

print(f"  Tree A: root_hash={mt_a.root_hash[:16]}..., size={mt_a.size}")
print(f"  Tree B: root_hash={mt_b.root_hash[:16]}..., size={mt_b.size}")

# Diff
d = bench("merkle_diff", merkle_diff, mt_a, mt_b)
print(f"  Merkle diff: {d}")

# Identical trees should have same hash
mt_c = MerkleTree.from_records(recs_a, key="id")
assert mt_a.root_hash == mt_c.root_hash, "Identical records should produce same hash!"
print(f"  Hash consistency check: PASSED")

# Scale
print("\n── Scale: Merkle tree construction ──")
def merkle_scale(n):
    recs = [{"id": i, "data": f"payload_{i}"} for i in range(n)]
    return MerkleTree.from_records(recs, key="id")

sizes = [5000 * SM, 20000 * SM, 100000 * SM]
scale_bench("merkle_build", merkle_scale, sizes)

print("\n✅ Section 14 complete.")

## 15. Gossip Protocol

Test `GossipState` — update, get, to_dict, size, merge. Convergence test with N nodes. Scale updates.

In [ ]:
# ── Section 15: Gossip ────────────────────────────────────────────────────────
from crdt_merge import GossipState

# Basic operations
gs1 = GossipState("node_1")
gs1.update("config_version", "1.0")
gs1.update("status", "active")
gs2 = GossipState("node_2")
gs2.update("config_version", "2.0")
gs2.update("region", "us-west")

print(f"  gs1 get('config_version'): {gs1.get('config_version')}")
print(f"  gs1 to_dict: {gs1.to_dict()}")
print(f"  gs1 size: {gs1.size}")
print(f"  gs2 size: {gs2.size}")

merged_gs = bench("gossip_merge", gs1.merge, gs2)
print(f"  Merged to_dict: {merged_gs.to_dict()}")
print(f"  Merged size: {merged_gs.size}")

# Convergence test with N nodes
print("\n── Convergence: N nodes ──")
N_NODES = 10 * SM
nodes = [GossipState(f"node_{i}") for i in range(N_NODES)]

# Each node updates its own key
for i, node in enumerate(nodes):
    node.update(f"key_{i}", f"value_{i}")

# Gossip rounds to converge
rounds = 0
converged = False
for _ in range(50):
    rounds += 1
    for i in range(len(nodes) - 1):
        nodes[i + 1] = nodes[i + 1].merge(nodes[i])
    for i in range(len(nodes) - 1, 0, -1):
        nodes[i - 1] = nodes[i - 1].merge(nodes[i])
    # Check convergence
    sizes = set(n.size for n in nodes)
    if len(sizes) == 1 and list(sizes)[0] >= N_NODES:
        converged = True
        break

print(f"  {N_NODES} nodes converged in {rounds} rounds: {converged}")
print(f"  Final node sizes: {[n.size for n in nodes[:5]]}...")
ALL_RESULTS["gossip_convergence"] = {
    "nodes": N_NODES, "rounds": rounds, "converged": converged, "status": "OK"
}

# Scale updates
print("\n── Scale: GossipState updates ──")
def gossip_scale(n):
    gs = GossipState("stress")
    for i in range(n):
        gs.update(f"k_{i}", f"v_{i}")
    return gs

sizes_gs = [1000 * SM, 5000 * SM, 20000 * SM]
scale_bench("gossip_updates", gossip_scale, sizes_gs)

print("\n✅ Section 15 complete.")

## 16. Arrow Merge

Test `ArrowMerge()` (no key in constructor). Compare with pandas merge. Scale to 5M×SM.

In [ ]:
# ── Section 16: Arrow ─────────────────────────────────────────────────────────
import pyarrow as pa
from crdt_merge import ArrowMerge, merge

am = ArrowMerge()

# Basic Arrow merge
ta = pa.table({"id": [1, 2, 3], "val": ["a", "b", "c"]})
tb = pa.table({"id": [2, 3, 4], "val": ["B", "C", "D"]})
result = bench("arrow_merge_basic", am.merge, ta, tb, key="id")
print(f"  Arrow merge result: {result.num_rows} rows")
print(result.to_pandas())

# Compare Arrow vs Pandas at scale
print("\n── Arrow vs Pandas comparison ──")
compare_results = {}
for n in [50000 * SM, 500000 * SM, 5000000 * SM]:
    ids_a = np.arange(n)
    ids_b = np.arange(n // 2, n + n // 2)
    vals_a = np.random.randn(n)
    vals_b = np.random.randn(n)
    now_ts = time.time()

    # Arrow merge
    ta = pa.table({"id": ids_a, "val": vals_a})
    tb = pa.table({"id": ids_b, "val": vals_b})
    t0 = time.perf_counter()
    r_arrow = am.merge(ta, tb, key="id")
    arrow_time = time.perf_counter() - t0

    # Pandas merge
    da = pd.DataFrame({"id": ids_a, "val": vals_a, "ts": [now_ts - 100]*n})
    db = pd.DataFrame({"id": ids_b, "val": vals_b, "ts": [now_ts]*n})
    t0 = time.perf_counter()
    r_pandas = merge(da, db, key="id", timestamp_col="ts")
    pandas_time = time.perf_counter() - t0

    speedup = pandas_time / arrow_time if arrow_time > 0 else float('inf')
    compare_results[n] = {"arrow_s": round(arrow_time, 4),
                          "pandas_s": round(pandas_time, 4),
                          "speedup": round(speedup, 2)}
    print(f"  n={n:>10,}: Arrow={arrow_time:.4f}s  Pandas={pandas_time:.4f}s  "
          f"Speedup={speedup:.2f}×")

ALL_RESULTS["arrow_vs_pandas"] = compare_results

print("\n✅ Section 16 complete.")

## 17. Async Merge

Test `amerge()` with `nest_asyncio` for Colab/Jupyter. Run concurrent merges.

In [ ]:
# ── Section 17: Async ─────────────────────────────────────────────────────────
import asyncio
import nest_asyncio
nest_asyncio.apply()

from crdt_merge import amerge

now = time.time()
df_a = pd.DataFrame({"id": [1, 2, 3], "val": ["a", "b", "c"],
                      "ts": [now - 100]*3})
df_b = pd.DataFrame({"id": [2, 3, 4], "val": ["B", "C", "D"],
                      "ts": [now]*3})

async def run_amerge():
    return await amerge(df_a, df_b, key="id")

loop = asyncio.get_event_loop()
result = bench("amerge_basic", loop.run_until_complete, run_amerge())
print(f"  amerge result shape: {result.shape}")
print(result)

# Concurrent merges
print("\n── Concurrent amerges ──")
async def concurrent_merges(n_tasks):
    tasks = []
    for i in range(n_tasks):
        a = pd.DataFrame({"id": range(100), "val": np.random.randn(100),
                          "ts": [now - 100]*100})
        b = pd.DataFrame({"id": range(50, 150), "val": np.random.randn(100),
                          "ts": [now]*100})
        tasks.append(amerge(a, b, key="id"))
    results = await asyncio.gather(*tasks)
    return results

n_concurrent = 10 * SM
t0 = time.perf_counter()
results = loop.run_until_complete(concurrent_merges(n_concurrent))
elapsed = time.perf_counter() - t0
print(f"  {n_concurrent} concurrent merges: {elapsed:.4f}s")
print(f"  Avg per merge: {elapsed / n_concurrent * 1000:.2f}ms")
ALL_RESULTS["amerge_concurrent"] = {
    "n_tasks": n_concurrent, "total_s": round(elapsed, 4),
    "avg_ms": round(elapsed / n_concurrent * 1000, 2), "status": "OK"
}

print("\n✅ Section 17 complete.")

## 18. Parallel Merge

Test `parallel_merge()` with two DataFrames. Compare sequential vs parallel.

In [ ]:
# ── Section 18: Parallel ──────────────────────────────────────────────────────
from crdt_merge import parallel_merge, merge

now = time.time()
N = 100000 * SM

left = pd.DataFrame({
    "id": range(N),
    "value": np.random.randn(N),
    "ts": [now - 100] * N
})
right = pd.DataFrame({
    "id": range(N // 2, N + N // 2),
    "value": np.random.randn(N),
    "ts": [now] * N
})

# Parallel merge
result_par = bench("parallel_merge", parallel_merge, left, right, key="id")
print(f"  Parallel merge result: {result_par.shape}")

# Sequential merge for comparison
result_seq = bench("sequential_merge", merge, left, right, key="id",
                   timestamp_col="ts")
print(f"  Sequential merge result: {result_seq.shape}")

par_time = ALL_RESULTS["parallel_merge"]["duration_s"]
seq_time = ALL_RESULTS["sequential_merge"]["duration_s"]
speedup = seq_time / par_time if par_time > 0 else float('inf')
print(f"\n  Parallel speedup: {speedup:.2f}× (seq={seq_time:.4f}s, par={par_time:.4f}s)")
ALL_RESULTS["parallel_vs_sequential"] = {
    "n": N, "sequential_s": seq_time, "parallel_s": par_time,
    "speedup": round(speedup, 2), "status": "OK"
}

print("\n✅ Section 18 complete.")

## 19. Integration Pipelines

5 pipelines chaining multiple crdt-merge subsystems end-to-end.

In [ ]:
# ── Section 19: Integration Pipelines ─────────────────────────────────────────
from crdt_merge import (
    evolve_schema, merge, dedup, MerkleTree, GCounter, VectorClock,
    serialize, deserialize, merge_stream, merge_dicts, MergeSchema,
    LWW, MaxWins, merge_with_provenance, export_provenance, GossipState
)

now = time.time()

# ── Pipeline 1: Schema evolve → merge → dedup → Merkle ──
print("── Pipeline 1: Schema → Merge → Dedup → Merkle ──")
t0 = time.perf_counter()

old_schema = {"id": "int", "name": "string"}
new_schema = {"id": "int", "name": "string", "email": "string"}
evo = evolve_schema(old_schema, new_schema)
print(f"  Schema evolved: {evo.resolved_schema}")

df_a = pd.DataFrame({"id": [1,2,3], "name": ["A","B","C"], "ts": [now-100]*3})
df_b = pd.DataFrame({"id": [2,3,4], "name": ["B2","C2","D"], "ts": [now]*3})
merged = merge(df_a, df_b, key="id", timestamp_col="ts")

names = merged["name"].tolist()
unique_names, removed = dedup(names)
print(f"  Dedup: {len(names)} -> {len(unique_names)} names")

recs = merged.to_dict("records")
mt = MerkleTree.from_records(recs, key="id")
print(f"  Merkle root: {mt.root_hash[:16]}..., size={mt.size}")

p1_time = time.perf_counter() - t0
ALL_RESULTS["pipeline_1"] = {"elapsed_s": round(p1_time, 4), "status": "OK"}
print(f"  Pipeline 1: {p1_time:.4f}s")

# ── Pipeline 2: GCounter aggregate → serialize → VectorClock ──
print("\n── Pipeline 2: GCounter → Serialize → VectorClock ──")
t0 = time.perf_counter()

counters = []
for i in range(20):
    gc = GCounter(f"node_{i}")
    gc.increment(f"node_{i}", amount=i + 1)
    counters.append(gc)

# Aggregate all
agg = counters[0]
for c in counters[1:]:
    agg = agg.merge(c)
print(f"  Aggregated GCounter value: {agg.value}")

data = serialize(agg)
restored = deserialize(data)
print(f"  Serialized & restored: value={restored.value}")

vc = VectorClock()
for i in range(20):
    vc.increment(f"node_{i}")
print(f"  VectorClock: {vc.to_dict()}")

p2_time = time.perf_counter() - t0
ALL_RESULTS["pipeline_2"] = {"elapsed_s": round(p2_time, 4), "status": "OK"}
print(f"  Pipeline 2: {p2_time:.4f}s")

# ── Pipeline 3: Stream merge → dedup strings ──
print("\n── Pipeline 3: Stream → Dedup ──")
t0 = time.perf_counter()

src_a = [{"id": i, "text": f"doc_{i}"} for i in range(500)]
src_b = [{"id": i + 250, "text": f"doc_{i + 250}"} for i in range(500)]
gen = merge_stream(iter(src_a), iter(src_b), key="id")
stream_results = list(gen)

# Flatten batches
all_texts = []
for batch in stream_results:
    if isinstance(batch, list):
        all_texts.extend([r.get("text", "") for r in batch])
    elif isinstance(batch, dict):
        all_texts.append(batch.get("text", ""))

unique_texts, removed_idx = dedup(all_texts)
print(f"  Stream produced {len(all_texts)} texts, dedup -> {len(unique_texts)}")

p3_time = time.perf_counter() - t0
ALL_RESULTS["pipeline_3"] = {"elapsed_s": round(p3_time, 4), "status": "OK"}
print(f"  Pipeline 3: {p3_time:.4f}s")

# ── Pipeline 4: Gossip convergence → merge_dicts ──
print("\n── Pipeline 4: Gossip → merge_dicts ──")
t0 = time.perf_counter()

gs_nodes = [GossipState(f"n_{i}") for i in range(5)]
for i, g in enumerate(gs_nodes):
    g.update(f"config_{i}", f"v{i}")
# Converge
for _ in range(10):
    for i in range(len(gs_nodes) - 1):
        gs_nodes[i + 1] = gs_nodes[i + 1].merge(gs_nodes[i])
    for i in range(len(gs_nodes) - 1, 0, -1):
        gs_nodes[i - 1] = gs_nodes[i - 1].merge(gs_nodes[i])

state = gs_nodes[0].to_dict()
extra = {"config_extra": "bonus_value"}
final = merge_dicts(state if isinstance(state, dict) else {}, extra)
print(f"  Gossip converged state + extra: {len(final)} keys")

p4_time = time.perf_counter() - t0
ALL_RESULTS["pipeline_4"] = {"elapsed_s": round(p4_time, 4), "status": "OK"}
print(f"  Pipeline 4: {p4_time:.4f}s")

# ── Pipeline 5: Full stack ──
print("\n── Pipeline 5: Full Stack ──")
t0 = time.perf_counter()

# Step 1: evolve_schema
old_s = {"id": "int", "name": "string", "score": "float"}
new_s = {"id": "int", "name": "string", "score": "double", "badge": "string"}
evo = evolve_schema(old_s, new_s)
print(f"  1. Schema: {evo.resolved_schema}")

# Step 2: MergeSchema merge
schema = MergeSchema(default=LWW(), score=MaxWins())
df_a = pd.DataFrame({"id": [1,2,3], "name": ["A","B","C"], "score": [80,90,70],
                      "ts": [now-100]*3})
df_b = pd.DataFrame({"id": [2,3,4], "name": ["B2","C2","D"], "score": [95,65,88],
                      "ts": [now]*3})
merged = merge(df_a, df_b, key="id", timestamp_col="ts", schema=schema)
print(f"  2. Merged: {merged.shape}")

# Step 3: Provenance
result_df, prov = merge_with_provenance(df_a, df_b, key="id", timestamp_col="ts")
prov_json = export_provenance(prov, format="json")
print(f"  3. Provenance conflicts: {prov.total_conflicts}")

# Step 4: Merkle
recs = result_df if isinstance(result_df, list) else result_df.to_dict("records")
mt = MerkleTree.from_records(recs, key="id")
print(f"  4. Merkle root: {mt.root_hash[:16]}..., size={mt.size}")

# Step 5: VectorClock
vc = VectorClock()
vc.increment("pipeline_node")
vc.increment("pipeline_node")
print(f"  5. VectorClock: {vc.to_dict()}")

p5_time = time.perf_counter() - t0
ALL_RESULTS["pipeline_5_fullstack"] = {"elapsed_s": round(p5_time, 4), "status": "OK"}
print(f"  Pipeline 5: {p5_time:.4f}s")

print("\n✅ Section 19 complete.")

## 20. Export Results to JSON

Save all benchmark results with metadata to `crdt_merge_v060_a100_benchmark.json`.

In [ ]:
# ── Section 20: Results JSON Export ───────────────────────────────────────────
import json, datetime

output = {
    "meta": {
        "version": "crdt-merge 0.6.0",
        "notebook": "A100 Stress Test",
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "hardware": {
            "gpu": GPU_NAME,
            "gpu_mem_gb": round(GPU_MEM_GB, 1),
            "ram_gb": round(RAM_GB, 1),
            "cpus": CPU_COUNT,
            "tier": TIER,
            "scale_multiplier": SM
        }
    },
    "results": ALL_RESULTS,
    "summary": {
        "total_benchmarks": len(ALL_RESULTS),
        "all_passed": all(
            v.get("status") == "OK" for v in ALL_RESULTS.values()
            if isinstance(v, dict) and "status" in v
        )
    }
}

out_path = "crdt_merge_v060_a100_benchmark.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2, default=str)

print(f"📁 Results saved to: {out_path}")
print(f"📊 Total benchmarks: {len(ALL_RESULTS)}")
print(f"✅ All passed: {output['summary']['all_passed']}")
print(f"\nSample keys: {list(ALL_RESULTS.keys())[:10]}...")

print("\n✅ Section 20 complete.")

## 21. Visualization — Throughput Charts

3×3 throughput grid and Arrow vs Pandas comparison bar chart.

In [ ]:
# ── Section 21: Charts ────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Collect scale benchmarks for throughput grid
scale_keys = [k for k in ALL_RESULTS if k.endswith("_scale")]
# Pick up to 9 for 3x3 grid
grid_keys = scale_keys[:9]

if grid_keys:
    rows = 3
    cols = 3
    fig, axes = plt.subplots(rows, cols, figsize=(18, 14))
    fig.suptitle("crdt-merge v0.6.0 — Throughput Scaling (A100 Stress Test)",
                 fontsize=16, fontweight="bold")

    for idx, key in enumerate(grid_keys):
        ax = axes[idx // cols][idx % cols]
        data = ALL_RESULTS[key]
        if isinstance(data, list) and len(data) > 0:
            ns = [d["n"] for d in data]
            tps = [d["throughput"] for d in data]
            ax.plot(ns, tps, "o-", linewidth=2, markersize=8, color="tab:blue")
            ax.set_title(key.replace("_scale", ""), fontsize=11, fontweight="bold")
            ax.set_xlabel("n")
            ax.set_ylabel("ops/s")
            ax.ticklabel_format(style="scientific", axis="both", scilimits=(0, 0))
            ax.grid(True, alpha=0.3)

    # Hide unused subplots
    for idx in range(len(grid_keys), rows * cols):
        axes[idx // cols][idx % cols].set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig("throughput_grid.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("📊 Saved: throughput_grid.png")
else:
    print("⚠️  No scale benchmarks found for throughput grid.")

# Arrow vs Pandas bar chart
if "arrow_vs_pandas" in ALL_RESULTS:
    avp = ALL_RESULTS["arrow_vs_pandas"]
    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig2.suptitle("Arrow vs Pandas Merge Performance", fontsize=14, fontweight="bold")

    labels = [f"{int(n):,}" for n in avp.keys()]
    arrow_times = [v["arrow_s"] for v in avp.values()]
    pandas_times = [v["pandas_s"] for v in avp.values()]
    speedups = [v["speedup"] for v in avp.values()]

    x = range(len(labels))
    width = 0.35
    ax1.bar([i - width/2 for i in x], arrow_times, width, label="Arrow", color="tab:green")
    ax1.bar([i + width/2 for i in x], pandas_times, width, label="Pandas", color="tab:orange")
    ax1.set_xlabel("Rows")
    ax1.set_ylabel("Time (s)")
    ax1.set_title("Merge Time")
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(labels, rotation=30)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis="y")

    ax2.bar(labels, speedups, color="tab:blue")
    ax2.set_xlabel("Rows")
    ax2.set_ylabel("Speedup (×)")
    ax2.set_title("Arrow Speedup over Pandas")
    ax2.axhline(y=1.0, color="red", linestyle="--", alpha=0.5)
    ax2.set_xticklabels(labels, rotation=30)
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig("arrow_vs_pandas.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("📊 Saved: arrow_vs_pandas.png")
else:
    print("⚠️  No Arrow vs Pandas data found.")

print("\n🏁 All 21 sections complete! Notebook stress test finished.")